In [ ]:
import pandas as pd
import re
from tqdm import tqdm  # Barra de progreso

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).upper()
    texto = texto.replace(chr(160), " ")
    texto = texto.replace("\r", " ")
    texto = texto.replace("\n", " ")
    texto = texto.replace("\t", " ")
    texto = texto.replace('"', "")

    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def obtener_palabras_validas(texto):
    STOP_WORDS = {"DE", "LA", "EL", "Y"}  # definida dentro

    palabras = texto.split(" ")

    return {
        p for p in palabras
        if len(p) > 1 and p not in STOP_WORDS
    }

import pandas as pd
from google.colab import files

uploaded = files.upload()

df = pd.read_excel(
    "MacroBuscar.xlsm",
    sheet_name="BUSQUEDA",
    engine="openpyxl",
    header=None
)

col_buscar = df.iloc[:, 0].fillna("").astype(str)
col_base = df.iloc[:, 1].fillna("").astype(str)

print("Archivo cargado correctamente")
print("Filas:", len(df))

# Normalizar columna base
base_normalizada = [normalizar_texto(x) for x in col_base]

# Precalcular palabras válidas para buscar
buscar_palabras = [
    obtener_palabras_validas(normalizar_texto(x))
    for x in col_buscar
]

resultados = []

for palabras in tqdm(buscar_palabras):

    coincidencias_fila = []

    if len(palabras) == 0:
        resultados.append(coincidencias_fila)
        continue

    for texto_base, texto_original in zip(base_normalizada, col_base):

        coincidencias = sum(
            1 for palabra in palabras
            if palabra in texto_base
        )

        porcentaje = coincidencias / len(palabras)

        if porcentaje >= 0.8:
            coincidencias_fila.append(
                f"{texto_original} ({porcentaje:.0%})"
            )

    resultados.append(coincidencias_fila)

    max_coinc = max(len(r) for r in resultados)

for i in range(max_coinc):
    df[f"Resultado_{i+1}"] = [
        r[i] if i < len(r) else ""
        for r in resultados
    ]

df.to_excel("resultado.xlsx", index=False)

Saving MacroBuscar.xlsm to MacroBuscar (1).xlsm
Archivo cargado correctamente
Filas: 1015


100%|██████████| 1015/1015 [00:02<00:00, 454.07it/s]


In [ ]:
from google.colab import files
files.download("resultado.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install pandas openpyxl httpx beautifulsoup4 ddgs rapidfuzz tqdm nest_asyncio

from google.colab import files
uploaded = files.upload()

import pandas as pd
import httpx
import asyncio
import nest_asyncio
from bs4 import BeautifulSoup
from ddgs import DDGS
from rapidfuzz import process
from tqdm import tqdm

nest_asyncio.apply()

archivo = "base_prov.xlsx"

df = pd.read_excel(archivo)

# lista básica de categorías
categorias = [
"logística",
"transporte",
"empaques",
"plásticos",
"cartón",
"metalmecánica",
"químicos",
"tecnología",
"ingeniería",
"mantenimiento industrial",
"materias primas",
"servicios industriales",
"consultoría",
"software"
]

# ciudades principales
ciudades_colombia = [
"Bogotá","Medellín","Cali","Barranquilla","Cartagena",
"Bucaramanga","Pereira","Manizales","Ibagué","Pasto",
"Cúcuta","Villavicencio","Neiva","Armenia","Santa Marta"
]


def buscar_urls(nombre):

    urls = []

    try:
        with DDGS() as ddgs:

            resultados = ddgs.text(nombre + " empresa", max_results=4)

            for r in resultados:
                urls.append(r["href"])

    except:
        pass

    return urls


async def scrapear(url, client):

    try:

        r = await client.get(url, timeout=10)

        soup = BeautifulSoup(r.text, "html.parser")

        for s in soup(["script","style"]):
            s.extract()

        texto = soup.get_text(" ", strip=True)

        return texto

    except:
        return ""


def detectar_ciudades(texto):

    texto = texto.lower()

    encontradas = []

    for c in ciudades_colombia:

        if c.lower() in texto:
            encontradas.append(c)

    return list(set(encontradas))


def detectar_categoria(texto):

    texto = texto.lower()

    match = process.extractOne(texto, categorias)

    if match and match[1] > 60:
        return match[0]

    return ""


async def procesar_proveedor(index, proveedor, client):

    urls = buscar_urls(proveedor)

    texto_total = ""

    for url in urls:

        texto = await scrapear(url, client)

        if len(texto) > 300:

            texto_total += texto[:2000]

    categoria = detectar_categoria(texto_total)

    ciudades = detectar_ciudades(texto_total)

    productos = texto_total[:200]

    servicios = texto_total[200:400]

    return index, categoria, productos, servicios, ", ".join(ciudades)


async def main():

    async with httpx.AsyncClient(headers={"User-Agent":"Mozilla/5.0"}) as client:

        tasks = []

        for i,row in df.iterrows():

            proveedor = str(row["proveedor"])

            tasks.append(
                procesar_proveedor(i, proveedor, client)
            )

        resultados = []

        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):

            resultados.append(await f)

        for r in resultados:

            i,categoria,productos,servicios,ciudades = r

            df.at[i,"categoria"] = categoria
            df.at[i,"productos_principales"] = productos
            df.at[i,"servicios_principales"] = servicios
            df.at[i,"ciudades"] = ciudades


await main()

df.to_excel("proveedores_completos.xlsx", index=False)

print("Proceso terminado")

files.download("proveedores_completos.xlsx")